# Chạy cái này trên kaggle trước để có remote model

In [1]:
!pip install fastapi uvicorn pydantic pytorch-lightning -q

In [2]:
import numpy as np
import torch
from torch import concat, diag, logical_and, logical_or, nn, tensor, tile
from torch.nn import Dropout
import pytorch_lightning as py_light
from functools import partial

# Model definitions

In [3]:
def multiply_head_with_embedding(prediction_head, embeddings):
    return prediction_head.matmul(embeddings.transpose(-1, -2))

def bce_loss(pos_logits, neg_logits, mask, epsilon=1e-10, device="cpu"):
    loss = torch.log(1. + torch.exp(-pos_logits) + epsilon) + torch.log(1. + torch.exp(neg_logits) + epsilon).mean(-1, keepdim=True)
    return (loss * mask.unsqueeze(-1)).sum() / mask.sum()

class DynamicPositionEmbedding(torch.nn.Module):
    def __init__(self, max_len, dimension):
        super(DynamicPositionEmbedding, self).__init__()
        self.max_len = max_len
        self.embedding = nn.Embedding(max_len, dimension)
        self.pos_indices = torch.arange(0, self.max_len, dtype=torch.int)
        self.register_buffer('pos_indices_const', self.pos_indices)

    def forward(self, x, device='cpu'):
        seq_len = x.shape[1]
        return self.embedding(self.pos_indices_const[-seq_len:]) + x

class SASRec(py_light.LightningModule):
    def __init__(self, hidden_size, dropout_rate, max_len, num_items, batch_size,
                 sampling_style, topk_sampling=False, topk_sampling_k=1000,
                 learning_rate=0.001, num_layers=2, loss='bce', bpr_penalty=None,
                 optimizer='adam', output_bias=False, share_embeddings=True,
                 final_activation=False):
        super(SASRec, self).__init__()
        self.learning_rate = learning_rate
        self.hidden_size = hidden_size
        self.dropout_rate = dropout_rate
        self.num_items = num_items
        self.batch_size = batch_size
        self.num_layers = num_layers
        self.max_len = max_len
        self.output_bias = output_bias
        self.share_embeddings = share_embeddings
        self.future_mask = torch.triu(torch.ones(max_len, max_len) * float('-inf'), diagonal=1)
        self.register_buffer('future_mask_const', self.future_mask)
        self.register_buffer('seq_diag_const', ~diag(torch.ones(max_len, dtype=torch.bool)))
        self.register_buffer('bias_ones', torch.ones([self.batch_size, self.max_len, 1]))
        if output_bias and share_embeddings:
            self.item_embedding = nn.Embedding(num_items + 1, hidden_size + 1, padding_idx=0)
        else:
            self.item_embedding = nn.Embedding(num_items + 1, hidden_size, padding_idx=0)
        self.positional_embedding_layer = DynamicPositionEmbedding(max_len, hidden_size)
        torch.nn.init.xavier_uniform_(self.item_embedding.weight.data)
        torch.nn.init.xavier_uniform_(self.positional_embedding_layer.embedding.weight.data)
        if share_embeddings:
            self.output_embedding = self.item_embedding
        elif (not share_embeddings) and output_bias:
            self.output_embedding = nn.Embedding(num_items + 1, hidden_size + 1, padding_idx=0)
        else:
            self.output_embedding = nn.Embedding(num_items + 1, hidden_size, padding_idx=0)
        self.norm = nn.LayerNorm([hidden_size])
        self.input_dropout = Dropout(dropout_rate)
        encoder_layer = nn.TransformerEncoderLayer(d_model=hidden_size, nhead=1,
                                                    dim_feedforward=hidden_size,
                                                    dropout=dropout_rate,
                                                    batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=self.num_layers, norm=self.norm)
        self.merge_attn_mask = True
        self.final_activation = nn.ELU(0.5) if final_activation else nn.Identity()
        self.loss_fn = loss
        if self.loss_fn == 'bce':
            self.loss = bce_loss
        else:
            raise ValueError('Loss function not supported')
        self.sampling_style = sampling_style
        self.topk_sampling = topk_sampling
        self.topk_sampling_k = topk_sampling_k
        self.optimizer = optimizer
        self.save_hyperparameters()

    def merge_attn_masks(self, padding_mask):
        batch_size = padding_mask.shape[0]
        seq_len = padding_mask.shape[1]
        if not self.merge_attn_mask:
            return self.future_mask_const[:seq_len, :seq_len]
        padding_mask_broadcast = ~padding_mask.bool().unsqueeze(1)
        future_masks = tile(self.future_mask_const[:seq_len, :seq_len], (batch_size, 1, 1))
        merged_masks = logical_or(padding_mask_broadcast, future_masks)
        diag_masks = tile(self.seq_diag_const[:seq_len, :seq_len], (batch_size, 1, 1))
        return logical_and(diag_masks, merged_masks)

    def forward(self, item_indices, mask):
        att_mask = self.merge_attn_masks(mask)
        items = self.item_embedding(item_indices)[:, :, :-1] if self.output_bias and self.share_embeddings \
                else self.item_embedding(item_indices)
        x = items * np.sqrt(self.hidden_size)
        x = self.positional_embedding_layer(x)
        x = self.encoder(self.input_dropout(x), att_mask)
        return concat([x, self.bias_ones], dim=-1) if self.output_bias else x

    def configure_optimizers(self):
        if self.optimizer == 'adam':
            return torch.optim.Adam(self.parameters(), lr=self.learning_rate)
        elif self.optimizer == 'adagrad':
            return torch.optim.Adagrad(self.parameters(), lr=self.learning_rate)
        raise ValueError('Optimizer not supported')

# Inference

In [5]:
class SASRecInference:
    def __init__(self, checkpoint_path, device=None):
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.device = device
        self.model = SASRec.load_from_checkpoint(checkpoint_path, map_location=device)
        self.model.eval()
        self.model.to(device)
        self.max_len = self.model.max_len
        self.num_items = self.model.num_items
        print(f"[SASRec] Loaded | device={device} | max_len={self.max_len} | num_items={self.num_items}")

    def _prepare_sequence(self, click_sequence: list) -> tuple:
        seq = click_sequence[-self.max_len:]
        seq_len = len(seq)
        pad_len = self.max_len - seq_len
        padded = [0] * pad_len + seq
        mask_arr = [0.0] * pad_len + [1.0] * seq_len
        item_indices = torch.tensor([padded], dtype=torch.long, device=self.device)
        mask = torch.tensor([mask_arr], dtype=torch.float, device=self.device)
        return item_indices, mask

    @torch.no_grad()
    def recommend_topk(self, click_sequence: list, k: int = 20, exclude_clicked: bool = True) -> tuple:
        item_indices, mask = self._prepare_sequence(click_sequence)
        x_hat = self.model.forward(item_indices, mask)
        last_hidden = x_hat[:, -1, :]
        logits = multiply_head_with_embedding(last_hidden, self.model.output_embedding.weight)
        logits = logits.squeeze(0)
        logits[0] = float("-inf")
        if exclude_clicked:
            for aid in click_sequence:
                if 1 <= aid <= self.num_items:
                    logits[aid] = float("-inf")
        actual_k = min(k, self.num_items)
        top_scores_tensor, top_indices_tensor = torch.topk(logits, k=actual_k)
        top_aids_shifted = top_indices_tensor.cpu().tolist()
        top_scores = top_scores_tensor.cpu().tolist()
        top_aids = [aid - 1 for aid in top_aids_shifted]  # về aid gốc
        return top_aids, top_scores

In [6]:
CHECKPOINT_PATH = "/kaggle/input/models/sandaria/sasrec/pytorch/otto-dataset/1/sasrec-otto-epoch4-recall_cutoff_200.308.ckpt"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
HOST = "0.0.0.0"
PORT = 8000

engine = SASRecInference(CHECKPOINT_PATH, device=DEVICE)

/tmp/ipykernel_57/2582544489.py:59: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=self.num_layers, norm=self.norm)


[SASRec] Loaded | device=cuda | max_len=50 | num_items=1855603


In [8]:
# test thử cái xem model chạy đc trên kaggle chưa
sample_aid_goc = [1492293,910862,1491172,424964,1515526,440486,109488,1507622,1734061,854637,718983,215311,718983,711125,50049,105393,959544,1734061,1842593,1464360,207905,1628317]
sample_aid_updated = [id + 1 for id in sample_aid_goc]

top_aids, top_scores = engine.recommend_topk(
        click_sequence=sample_aid_updated,
        k=20,
        exclude_clicked=True,
    )
print(f"Input clicks: {sample_aid_goc}")
print(f"Top-20: {top_aids}")
print(f"Scores: {[f'{s:.4f}' for s in top_scores[:5]]} ...")

Input clicks: [1492293, 910862, 1491172, 424964, 1515526, 440486, 109488, 1507622, 1734061, 854637, 718983, 215311, 718983, 711125, 50049, 105393, 959544, 1734061, 1842593, 1464360, 207905, 1628317]
Top-20: [556314, 1123537, 464673, 1007803, 1763549, 103974, 880854, 1491502, 1453277, 937893, 165854, 259724, 1245311, 606921, 1781681, 1532558, 434686, 1568162, 924095, 1483294]
Scores: ['14.0753', '13.9654', '13.2728', '13.0544', '12.9465'] ...


# FastAPI Server

In [9]:
import threading
import time
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import List, Optional

# request/response schemas 
class RecommendRequest(BaseModel):
    click_sequence: List[int] = Field(..., min_length=1, description="List aid gốc (chưa +1)")
    k: int = Field(20, ge=1, le=500, description="Số lượng gợi ý trả về")
    exclude_clicked: bool = Field(True, description="Loại bỏ item đã click khỏi kết quả")

class RecommendResponse(BaseModel):
    top_aids: List[int]
    top_scores: List[float]

app = FastAPI(title="SASRec Recommendation API")

# kiểm tra xem server đang hoạt động bình thường không
@app.get("/health")
def health():
    return {"status": "ok", "device": DEVICE,
            "max_len": engine.max_len, "num_items": engine.num_items}

@app.post("/recommend", response_model=RecommendResponse)
def recommend(req: RecommendRequest):
    try:
        # shift +1 vào aid đầu vào theo đúng yêu cầu sasrec
        shifted = [aid + 1 for aid in req.click_sequence]
        top_aids, top_scores = engine.recommend_topk(
            click_sequence=shifted,
            k=req.k,
            exclude_clicked=req.exclude_clicked,
        )
        return RecommendResponse(top_aids=top_aids, top_scores=top_scores)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# Start server and Cloudflare tunnel

In [ ]:
import os
import re
import subprocess

def start_cloudflare_tunnel(port):
    print("[CLOUDFLARE] Đang khởi tạo tunnel...")
    if not os.path.exists("./cloudflared"):
        print("[CLOUDFLARE] Đang tải cloudflared...")
        subprocess.run(["wget", "-q",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            "-O", "cloudflared"])
        subprocess.run(["chmod", "+x", "cloudflared"])

    proc = subprocess.Popen(
        ["./cloudflared", "tunnel", "--url", f"http://localhost:{port}"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in iter(proc.stdout.readline, ""):
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            url = match.group(0)
            print(f"\n{'='*60}")
            print(f"  SASRec API URL: {url}")
            print(f"  Docs:           {url}/docs")
            print(f"{'='*60}\n")
            return url, proc
    return None, proc

# Chạy Uvicorn trong thread riêng
server_thread = threading.Thread(
    target=lambda: uvicorn.run(app, host=HOST, port=PORT, log_level="warning"),
    daemon=True
)
server_thread.start()
time.sleep(3)

public_url, tunnel_proc = start_cloudflare_tunnel(PORT)

if public_url:
    while True:
        time.sleep(10)
else:
    print("[ERROR] Không thể khởi tạo Cloudflare Tunnel.")

ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use


[CLOUDFLARE] Đang khởi tạo tunnel...

  SASRec API URL: https://saturday-bruce-wyoming-varied.trycloudflare.com
  Docs:           https://saturday-bruce-wyoming-varied.trycloudflare.com/docs

